# Otimização de Joins com PySpark — YouTube

Neste notebook exploramos três abordagens para realizar o mesmo join entre `df_video` e `df_comments`:

1. **Baseline** — leitura simples, tabelas temporárias e join via Spark SQL
2. **Com repartition/coalesce** — controle manual do particionamento
3. **Otimizado** — seleção de colunas necessárias + broadcast hint + filtros antecipados + explain

Ao final comparamos os planos de execução e salvamos o join otimizado.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName('YouTube - Otimizacao') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'SparkSession iniciada | Spark {spark.version}')

---
## ▶ ETAPA 1 — Baseline: leitura simples + tabelas temporárias + join SQL

### Passo 1 — Leitura dos arquivos parquet

In [ ]:
# Passo 1a: Leitura do arquivo de vídeos preparados
df_video = spark.read.parquet('videos-preparados.snappy.parquet')

# Passo 1b: Leitura do arquivo de comentários tratados
df_comments = spark.read.parquet('video-comments-tratados.snappy.parquet')

print('=== df_video ===')
df_video.printSchema()
print(f'Registros: {df_video.count()} | Partições: {df_video.rdd.getNumPartitions()}')

print('\n=== df_comments ===')
df_comments.printSchema()
print(f'Registros: {df_comments.count()} | Partições: {df_comments.rdd.getNumPartitions()}')

### Passo 2 — Criação das tabelas temporárias

In [ ]:
# Passo 2: Registrar os dataframes como tabelas temporárias no Spark SQL
df_video.createOrReplaceTempView('tb_video')
df_comments.createOrReplaceTempView('tb_comments')

print('Tabelas temporárias criadas:')
spark.catalog.listTables()

### Passo 3 — Join via Spark SQL

In [ ]:
# Passo 3: Join entre as tabelas temporárias usando Spark SQL
# Selecionamos colunas sem ambiguidade renomeando as de df_comments com sufixo '_c'
join_video_comments = spark.sql("""
    SELECT
        v.`Video ID`,
        v.Title,
        v.`Published At`,
        v.Keyword,
        v.Likes,
        v.Comments,
        v.Views,
        v.Interaction,
        v.Year,
        v.Month,
        v.`Keyword Index`,
        c.Comment,
        c.Sentiment,
        c.`Likes Comment`
    FROM tb_video  AS v
    LEFT JOIN tb_comments AS c
        ON v.`Video ID` = c.`Video ID`
""")

print(f'Registros no join: {join_video_comments.count()}')
join_video_comments.show(5, truncate=True)

---
## ▶ ETAPA 2 — Com repartition e coalesce

### Passo 1 — Leitura com repartition

In [ ]:
# repartition(n) redistribui os dados em n partições via shuffle completo.
# Útil para aumentar paralelismo antes de joins ou operações pesadas.
df_video_rep = spark.read \
    .parquet('videos-preparados.snappy.parquet') \
    .repartition(8, '`Video ID`')

df_comments_rep = spark.read \
    .parquet('video-comments-tratados.snappy.parquet') \
    .repartition(8, '`Video ID`')

print(f'df_video_rep   partições após repartition: {df_video_rep.rdd.getNumPartitions()}')
print(f'df_comments_rep partições após repartition: {df_comments_rep.rdd.getNumPartitions()}')

### Passo 2 — Tabelas temporárias com repartition

In [ ]:
# Registramos os dataframes reparticionados como novas tabelas temporárias
df_video_rep.createOrReplaceTempView('tb_video_rep')
df_comments_rep.createOrReplaceTempView('tb_comments_rep')

print('Tabelas temporárias com repartition criadas:')
spark.catalog.listTables()

### Passo 3 — Join via Spark SQL com repartition

In [ ]:
# Join usando as tabelas reparticionadas pela chave de join ('Video ID').
# Isso pode reduzir o shuffle durante o join, pois os dados já estão co-particionados.
join_video_comments_rep = spark.sql("""
    SELECT
        v.`Video ID`,
        v.Title,
        v.`Published At`,
        v.Keyword,
        v.Likes,
        v.Comments,
        v.Views,
        v.Interaction,
        v.Year,
        v.Month,
        v.`Keyword Index`,
        c.Comment,
        c.Sentiment,
        c.`Likes Comment`
    FROM tb_video_rep    AS v
    LEFT JOIN tb_comments_rep AS c
        ON v.`Video ID` = c.`Video ID`
""")

print(f'Registros no join com repartition: {join_video_comments_rep.count()}')
join_video_comments_rep.show(5, truncate=True)

### Passo 4 — Aplicando coalesce

In [ ]:
# coalesce(n) reduz o número de partições SEM shuffle completo (mais eficiente que repartition
# quando o objetivo é reduzir). Ideal para consolidar partições antes de salvar em disco.
join_coalesced = join_video_comments_rep.coalesce(4)

print(f'Partições após coalesce: {join_coalesced.rdd.getNumPartitions()}')
print(f'Registros: {join_coalesced.count()}')
join_coalesced.show(5, truncate=True)

---
## ▶ ETAPA 3 — Explain: comparando os planos de execução

In [ ]:
print('=' * 70)
print('PLANO DE EXECUÇÃO — Join BASELINE (sem repartition)')
print('=' * 70)
join_video_comments.explain(extended=True)

In [ ]:
print('=' * 70)
print('PLANO DE EXECUÇÃO — Join com REPARTITION + COALESCE')
print('=' * 70)
join_coalesced.explain(extended=True)

---
## ▶ ETAPA 4 — Join Otimizado: todas as técnicas combinadas

In [ ]:
# -----------------------------------------------------------------------
# OTIMIZAÇÃO 1: Projeção antecipada (column pruning)
# Selecionamos APENAS as colunas necessárias antes do join, reduzindo
# o volume de dados trafegados durante o shuffle.
# -----------------------------------------------------------------------
df_video_opt = spark.read \
    .parquet('videos-preparados.snappy.parquet') \
    .select(
        F.col('Video ID'),
        F.col('Title'),
        F.col('Published At'),
        F.col('Keyword'),
        F.col('Likes'),
        F.col('Comments'),
        F.col('Views'),
        F.col('Interaction'),
        F.col('Year'),
        F.col('Month'),
        F.col('Keyword Index')
    )

# -----------------------------------------------------------------------
# OTIMIZAÇÃO 2: Projeção antecipada em df_comments
# Selecionamos somente as colunas exclusivas de comentários + a chave de join.
# -----------------------------------------------------------------------
df_comments_opt = spark.read \
    .parquet('video-comments-tratados.snappy.parquet') \
    .select(
        F.col('Video ID'),
        F.col('Comment'),
        F.col('Sentiment'),
        F.col('Likes Comment')
    )

# -----------------------------------------------------------------------
# OTIMIZAÇÃO 3: Filtro de nulos antecipado (predicate pushdown)
# Removemos linhas inválidas ANTES do join, evitando processar dados
# desnecessários no shuffle e na fase de join.
# -----------------------------------------------------------------------
df_video_opt    = df_video_opt.filter(F.col('Video ID').isNotNull())
df_comments_opt = df_comments_opt.filter(F.col('Video ID').isNotNull())

# -----------------------------------------------------------------------
# OTIMIZAÇÃO 4: Reparticionamento pela chave de join (co-particionamento)
# Ao repartir ambos os DataFrames pela mesma chave ('Video ID'), garantimos
# que registros com o mesmo ID fiquem na mesma partição, eliminando o
# shuffle durante o join (Sort Merge Join torna-se mais eficiente).
# -----------------------------------------------------------------------
df_video_opt    = df_video_opt.repartition(8, F.col('Video ID'))
df_comments_opt = df_comments_opt.repartition(8, F.col('Video ID'))

# -----------------------------------------------------------------------
# OTIMIZAÇÃO 5: Cache dos DataFrames filtrados e reparticionados
# Como vamos usar esses DataFrames no join e potencialmente em outras
# operações, o cache evita releitura do disco a cada ação.
# -----------------------------------------------------------------------
df_video_opt.cache()
df_comments_opt.cache()

# Materializar o cache
cnt_v = df_video_opt.count()
cnt_c = df_comments_opt.count()
print(f'df_video_opt  em cache: {cnt_v} registros | {df_video_opt.rdd.getNumPartitions()} partições')
print(f'df_comments_opt em cache: {cnt_c} registros | {df_comments_opt.rdd.getNumPartitions()} partições')

In [ ]:
# -----------------------------------------------------------------------
# OTIMIZAÇÃO 6: Tabelas temporárias a partir dos DataFrames otimizados
# Registramos as versões enxutas e pré-processadas como views temporárias
# para utilizar no Spark SQL com todas as otimizações já aplicadas.
# -----------------------------------------------------------------------
df_video_opt.createOrReplaceTempView('tb_video_opt')
df_comments_opt.createOrReplaceTempView('tb_comments_opt')

print('Tabelas temporárias otimizadas registradas com sucesso.')

In [ ]:
# -----------------------------------------------------------------------
# OTIMIZAÇÃO 7: Join via Spark SQL com BROADCAST hint
# O hint /*+ BROADCAST(v) */ instrui o Spark a enviar o DataFrame menor
# (df_video_opt, ~1.869 registros) para cada executor, eliminando o shuffle
# do lado maior (df_comments_opt, ~18.409 registros).
# Isso converte um Sort Merge Join em um Broadcast Hash Join, muito mais rápido.
# -----------------------------------------------------------------------
join_video_comments_otimizado = spark.sql("""
    SELECT /*+ BROADCAST(v) */
        v.`Video ID`,
        v.Title,
        v.`Published At`,
        v.Keyword,
        v.Likes,
        v.Comments,
        v.Views,
        v.Interaction,
        v.Year,
        v.Month,
        v.`Keyword Index`,
        c.Comment,
        c.Sentiment,
        c.`Likes Comment`
    FROM tb_video_opt    AS v
    LEFT JOIN tb_comments_opt AS c
        ON v.`Video ID` = c.`Video ID`
""")

print(f'Registros no join otimizado: {join_video_comments_otimizado.count()}')
join_video_comments_otimizado.show(5, truncate=True)

In [ ]:
# -----------------------------------------------------------------------
# EXPLAIN do join otimizado
# Verificamos no plano físico se o Spark escolheu BroadcastHashJoin (ideal)
# em vez de SortMergeJoin. O extended=True mostra planos lógico e físico.
# -----------------------------------------------------------------------
print('=' * 70)
print('PLANO DE EXECUÇÃO — Join OTIMIZADO (broadcast + column pruning + cache)')
print('=' * 70)
join_video_comments_otimizado.explain(extended=True)

In [ ]:
# -----------------------------------------------------------------------
# OTIMIZAÇÃO 8: coalesce antes de salvar
# Reduzimos as partições para 4 sem shuffle adicional (coalesce é mais leve
# que repartition quando queremos diminuir partições), gerando menos arquivos
# parquet e escrita mais eficiente.
# -----------------------------------------------------------------------
join_otimizado_final = join_video_comments_otimizado.coalesce(4)

print(f'Partições antes de salvar: {join_otimizado_final.rdd.getNumPartitions()}')

# -----------------------------------------------------------------------
# Salvando o resultado otimizado no formato parquet
# Usamos snappy (padrão) como codec de compressão: bom equilíbrio entre
# velocidade de leitura/escrita e tamanho em disco.
# -----------------------------------------------------------------------
join_otimizado_final.write \
    .option('header', 'true') \
    .option('compression', 'snappy') \
    .mode('overwrite') \
    .parquet('join-videos-comments-otimizado')

print('Join otimizado salvo como join-videos-comments-otimizado com sucesso!')

# Verificação final
df_verificacao = spark.read.parquet('join-videos-comments-otimizado')
print(f'Registros verificados no arquivo salvo: {df_verificacao.count()}')
print(f'Colunas: {df_verificacao.columns}')
df_verificacao.show(5, truncate=True)

In [ ]:
# Liberar cache antes de encerrar
df_video_opt.unpersist()
df_comments_opt.unpersist()

spark.stop()
print('SparkSession encerrada com sucesso!')